# Install Packages

In [ ]:
# install required libraries, specific versions are needed due to conflicts
!mamba install --force-reinstall aiohttp -y
!pip install -U "xformers<0.0.26" --index-url https://download.pytorch.org/whl/cu121
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"

# Import Libraries and Prepare Environment

In [ ]:
import os
import random
import torch

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from unsloth import is_bfloat16_supported

from datasets import load_dataset, Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from transformers import AutoModel, AutoTokenizer

# wanDB require 3rd party authorisation
os.environ["WANDB_DISABLED"] = "true"

# Set Up Settings for the Model

In [2]:
max_seq_length = 4096 

# type will be selected automatically depending on GPU
dtype = None

# to save memory use 4 instead of 32
load_in_4bit = True

# Import model

In [3]:
def load_model_training(path):
    # load model from the Hugging Face portal with optimisations
    m, t = FastLanguageModel.from_pretrained(
        model_name = path,
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )

    # set up setting for LoRA optimisations
    m = FastLanguageModel.get_peft_model(
        m,
        r = 32,
        lora_alpha = 32,
        random_state = 110,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = False,
        use_rslora = False,
        loftq_config = None
    )
    
    return m, t

def load_model_inference(path):
    m, t = load_model_training(path)
    FastLanguageModel.for_inference(m)
    t = get_chat_template(
        t,
        chat_template = "phi-3",
        mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}, 
    )
    
    return m, t

def save_model(path):
    trainer.model.save_pretrained(path)
    trainer.tokenizer.save_pretrained(path)

# Load Data

In [4]:
# double check for empty values and remove if any
def filter_non_empty(row):
    return row['code'] not in (None, "") and row['test_code'] not in (None, "")

# for large dataset to sample only specific percentage
def sample_dataset(dataset, sample_ratio=0.1):
    sample_size = int(len(dataset) * sample_ratio)
    sampled_indices = random.sample(range(len(dataset)), sample_size)
    return dataset.select(sampled_indices)

# parsing csv sometimes add double //, this fix removes it
def unescape_string(s):
    return s.replace('\\n', '\n').replace('\\t', '\t')

# format input according to expected Phi 3 template
def formatting_function(examples):
    formatted_texts = [
        f"Human: {unescape_string(code)} GPT: {unescape_string(test_code)}"
        for code, test_code in zip(examples['code'], examples['test_code'])
    ]
    return {"formated_text": formatted_texts}

In [ ]:
# import model
model, tokenizer = load_model_training("unsloth/Phi-3-mini-4k-instruct")

# load and preprocess dataset
dataset_go_tests = load_dataset('csv', data_files='file.csv')
filtered_dataset = dataset_go_tests['train'].filter(filter_non_empty)
filtered_dataset = filtered_dataset.map(formatting_function, batched = True)
train_dataset, validation_dataset = filtered_dataset.train_test_split(test_size=0.20).values()

In [ ]:
validation_dataset = sample_dataset(validation_dataset)

# Create Trainer and Train

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = validation_dataset,
    dataset_text_field = "formated_text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        per_device_eval_batch_size = 1,
        gradient_accumulation_steps = 6,
        warmup_steps = 5,
        max_steps = 1600,
        evaluation_strategy="steps",
        eval_steps=500,
        fp16 = not is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        save_steps=200,
        save_total_limit=3,
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 113,
        output_dir = "outputs-large-4k",
        learning_rate = 2e-4,
    ),
)

In [ ]:
stats = trainer.train()
trainer.save_model("outputs/model-large-data-4k")
tokenizer.save_pretrained("outputs/model-large-data-4k")

In [25]:
def get_answer_from_model(m, t, extra_command, function_names):
    if isinstance(function_names, str):
        function_names = [function_names]

    function_tests_request = "Don't write any text only Go code. Generate unit tests in Go for functions named: "
    function_tests_request += ', '.join(f"'{fn}'" for fn in function_names)
    function_tests_request += ". Each test should be unique and test different scenarios, such as input validation, successful and failure modes, and edge cases."

    if extra_command:
        function_tests_request += " " + extra_command

    messages = [
        {"from": "human", "value": function_tests_request}
    ]
    
    i = t.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")
    
    outputs = m.generate(input_ids = i, max_new_tokens = 2000, use_cache = True, no_repeat_ngram_size=20, num_return_sequences=1, temperature=0.8)
    return t.batch_decode(outputs)

def print_answers(answers):
    for output in answers:
        print(output)

In [ ]:
print("Model after training 0 steps")
model_clean, tokenizer_clean = load_model_inference("unsloth/Phi-3-mini-4k-instruct")

print("Model after training 1600 steps 4k")
model_1600, tokenizer_1600 = load_model_inference("/kaggle/working/outputs/model-large-data-4k")
